# Data Balancing Pipeline for Vishing Detection (Augmented Data)

This notebook implements a robust pipeline to counteract class imbalance using the **augmented dataset (1M+ records)**, generating new datasets using various techniques:
- **Random Oversampling**
- **SMOTE** (Synthetic Minority Over-sampling Technique)
- **Borderline SMOTE**
- **SMOTE + Undersampling** (Hybrid)

We will generate versions with **10%**, **20%**, and **25%** proportions of the minority class (`is_vishing`). The outputs will be saved to the path `data/balanced/augmented`, optimized in Parquet format.

## 1. Import Libraries and Configure Environment

In [1]:
import os
import pandas as pd
import numpy as np

# Imblearn - Oversampling and Hybrids
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import pyarrow

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load and Prepare Augmented Dataset

In [2]:
RAW_DATA_PATH = 'data/augmented_data/dataset_1M_vishing_ctgan.parquet'
df_original = pd.read_parquet(RAW_DATA_PATH)

# We will exclude features that the previous notebook flagged
exclude_cols = ['session_id', 'customer_id', 'session_timestamp', 'device_type', 'claim_category', 
                'biocatch_risk_score', 'biocatch_genuine_score', 'biocatch_ato_indicator', 
                'biocatch_social_eng_indicator', 'biocatch_bot_indicator', 'os_type', 'app_version']

target_col = 'is_vishing'

features = [c for c in df_original.columns if c not in exclude_cols and c != target_col]

X = df_original[features].fillna(0) # Simple fill n/a for processing
y = df_original[target_col]

majority_count = sum(y == 0)
minority_count = sum(y == 1)
total_count = len(y)

print(f"Total records: {total_count}")
print(f"Majority Class (Legitimate, 0): {majority_count} ({majority_count/total_count:.2%})")
print(f"Minority Class (Vishing,   1): {minority_count} ({minority_count/total_count:.2%})")

Total records: 1000000
Majority Class (Legitimate, 0): 985000 (98.50%)
Minority Class (Vishing,   1): 15000 (1.50%)


## 3. Define General Pipeline Functions

In [5]:
def get_sampling_strategy(pct, majority_samples):
    """
    Computes the ratio for imblearn (minority / majority) given a target percentage 
    for the minority class in the full set.
    Equation: N_min = (pct / (1 - pct)) * N_maj
    """
    ratio = pct / (1.0 - pct)
    return ratio

def save_dataset(X_res, y_res, technique, pct, base_dir='data/balanced/augmented'):
    """
    Takes resampled X and y matrices, concatenates them, and exports to the file system.
    Expected format: data/AD/technique/percentage.parquet
    """
    dir_path = os.path.join(base_dir, technique)
    os.makedirs(dir_path, exist_ok=True)
    
    # Format the file name (e.g. 10.parquet)
    pct_str = f"{int(pct * 100)}"
    file_path = os.path.join(dir_path, f"{pct_str}.parquet")
    
    df_out = pd.concat([X_res.copy(), y_res.copy()], axis=1)
    df_out.to_parquet(file_path, index=False)
    print(f"✅ Saved: {file_path} (Vishing distribution: {df_out['is_vishing'].mean():.2%}) | Total: {len(df_out)} rows")

TARGET_PERCENTAGES = [0.10, 0.20, 0.25]

## 4. Random Oversampling

In [6]:
print("Generating Random Oversampling...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    ros = RandomOverSampler(sampling_strategy=strategy, random_state=42)
    X_res, y_res = ros.fit_resample(X, y)
    save_dataset(X_res, y_res, 'random_oversampling', pct)

Generating Random Oversampling...
✅ Saved: data/balanced/augmented\random_oversampling\10.parquet (Vishing distribution: 10.00%) | Total: 1094444 rows
✅ Saved: data/balanced/augmented\random_oversampling\20.parquet (Vishing distribution: 20.00%) | Total: 1231250 rows
✅ Saved: data/balanced/augmented\random_oversampling\25.parquet (Vishing distribution: 25.00%) | Total: 1313333 rows


## 5. SMOTE

In [7]:
print("Generating SMOTE...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    smote = SMOTE(sampling_strategy=strategy, random_state=42)
    X_res, y_res = smote.fit_resample(X, y)
    save_dataset(X_res, y_res, 'smote', pct)

Generating SMOTE...
✅ Saved: data/balanced/augmented\smote\10.parquet (Vishing distribution: 10.00%) | Total: 1094444 rows
✅ Saved: data/balanced/augmented\smote\20.parquet (Vishing distribution: 20.00%) | Total: 1231250 rows
✅ Saved: data/balanced/augmented\smote\25.parquet (Vishing distribution: 25.00%) | Total: 1313333 rows


## 6. Borderline SMOTE

In [8]:
print("Generating Borderline SMOTE...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    bsmote = BorderlineSMOTE(sampling_strategy=strategy, random_state=42, kind='borderline-1')
    X_res, y_res = bsmote.fit_resample(X, y)
    save_dataset(X_res, y_res, 'borderline_smote', pct)

Generating Borderline SMOTE...
✅ Saved: data/balanced/augmented\borderline_smote\10.parquet (Vishing distribution: 10.00%) | Total: 1094444 rows
✅ Saved: data/balanced/augmented\borderline_smote\20.parquet (Vishing distribution: 20.00%) | Total: 1231250 rows
✅ Saved: data/balanced/augmented\borderline_smote\25.parquet (Vishing distribution: 25.00%) | Total: 1313333 rows


## 7. SMOTE + Undersampling (Pipeline)

In [9]:
print("Generating SMOTE + Undersampling...")
# Strategy: we raise Vishing drastically, and lower the majority class.
for pct in TARGET_PERCENTAGES:
    target_majority_count = int(majority_count * 0.9)  # Lower legitimate by 10%
    target_minority_count = int(target_majority_count * (pct / (1 - pct)))
    
    over = SMOTE(sampling_strategy={1: target_minority_count}, random_state=42)
    under = RandomUnderSampler(sampling_strategy={0: target_majority_count}, random_state=42)
    
    steps = [('o', over), ('u', under)]
    pipeline = Pipeline(steps=steps)
    
    X_res, y_res = pipeline.fit_resample(X, y)
    save_dataset(X_res, y_res, 'smote_undersampling', pct)

Generating SMOTE + Undersampling...
✅ Saved: data/balanced/augmented\smote_undersampling\10.parquet (Vishing distribution: 10.00%) | Total: 985000 rows
✅ Saved: data/balanced/augmented\smote_undersampling\20.parquet (Vishing distribution: 20.00%) | Total: 1108125 rows
✅ Saved: data/balanced/augmented\smote_undersampling\25.parquet (Vishing distribution: 25.00%) | Total: 1182000 rows
